# notebook 17 — 次元4の原理化：独立設定本数 p=4 を強制する原理はあるか

nb13〜nb16 で追加原理の地図が確定した。被覆原理 II（nb13）が**単一時間（1）と非コンパクト（R）**を
与件 cos2θ の内側（位相構造）から原理的に出す。一方 nb15 は、残る唯一の独立要素である
**空間次元3（⟺ 独立設定本数 p=4）**を与件から導く原理は無い、と否定的に確定した。

本 notebook は引き継ぎ E-1 の最重要 open「**次元4の原理化**」に正面から取り組む。攻略は
nb15 の到達点を踏まえ、**反証ハードルの高い順＝3→2→1** の三部構成で進める：

- **Part 3（否定の精密化）**：nb15 の否定は *素朴版* の否定だった。これを「測定論的補強を入れても
  p は枠内で一意化されない」という形に**精密化**し、Part 2/1 が *何を超えれば成功か* の判定基準を先に固める。
- **Part 2（立場B の精緻化）**：空間3を測定方向 SO(3) の3軸から取る立場B を、循環性まで厳密に詰める。
- **Part 1（測定論の一括基礎づけ）**：p の上限と被覆IIの動機を同じ測定論 st から出せるか（D-12）。

**最重要の規律（D-10/D-13/D-14）**：次元4は「p=4 と決めて時空4を後付け」「3＝色3」「4N の 4＝次元4」
といった**数の一致の誘惑**が最も強い領域である。各段で「これは値を*予言*したのか、値を*仮定*して
意味を後付けしたのか」を冷徹に自己点検する。空振りも nb15 同様、正直な結論として記録する。

## 17.0 セットアップ

In [1]:
import numpy as np
from numpy.linalg import eigh
np.set_printoptions(precision=4, suppress=True, linewidth=120)

N = 3  # 与件（QCD との同定）
print("setup done. N =", N)

setup done. N = 3


---
# Part 3（探索順①）：否定の精密化 — p を一意化する条件は枠内に無い

nb15 は「与件 cos2θ から p は出ない」を*素朴版*で否定した。ここでは否定を一段精密化する。
問いは「**測定論的補強を含めても、枠内で p=4 *だけ* が許される条件は構成できるか**」。
答えが「できない」なら、Part 2/1 が原理に格上げするには**枠の外**から要素を入れるしかない、
という判定基準が確定する。

## 17.1 与件の情報内容と p の所在

与件 `cos2(θi−θj)` は2測定方向の相対角 θ の1変数関数で、単一設定空間の相関は rank2
（`cos2θ, sin2θ` の2次元 = S¹ の1次元情報）。独立設定本数 p は「S¹ を何本直積するか」という、
この相関関数*そのものには現れない*選択である（nb04 で p は入力、nb15 と整合）。

In [2]:
m = 200
th = np.linspace(0, 2*np.pi, m, endpoint=False)
C1 = np.cos(2*(th[:, None] - th[None, :]))/(2*N)
ev = np.sort(np.abs(eigh(C1)[0]))[::-1]
rank = int((ev > 1e-9*ev[0]).sum())
print(f"単一設定空間の相関 C(θ,θ') の rank = {rank}（cos2θ, sin2θ の2次元）")
print("→ 1つの設定方向がもたらす内在次元は 1（S¹）。p は『S¹ を何本直積するか』。")
print("  これは C 自体には現れない（C は1本ぶんの相関）。p は外部入力。")

単一設定空間の相関 C(θ,θ') の rank = 2（cos2θ, sin2θ の2次元）
→ 1つの設定方向がもたらす内在次元は 1（S¹）。p は『S¹ を何本直積するか』。
  これは C 自体には現れない（C は1本ぶんの相関）。p は外部入力。


## 17.2 どの p も等しく無矛盾（一意化条件の不在）

p 本の独立設定で T^p を構成しても、p をどう選んでも内部矛盾は生じない。すなわち**どの p も
等しく許容される**＝枠内に p を一意化する条件が無い。各 p で MDS の内在次元が p に一致すること、
かつどの p も同格に構成できることを確認する。

In [3]:
def torus_intrinsic_dim(p, n_per=8):
    ph = np.linspace(0, 2*np.pi, n_per, endpoint=False)
    grids = np.meshgrid(*([ph]*p), indexing="ij")
    pts = np.stack([g.ravel() for g in grids], 1)
    M = pts.shape[0]
    d2 = np.zeros((M, M))
    for col in range(p):
        dphi = pts[:, col][:, None] - pts[:, col][None, :]
        a = np.abs(np.cos(2*dphi))/(2*N)
        with np.errstate(divide="ignore"):
            d = -np.log(a)
        d[~np.isfinite(d)] = 0.0; np.fill_diagonal(d, 0.0)
        d2 += d**2
    J = np.eye(M) - np.ones((M, M))/M
    B = -0.5*J@d2@J; B = 0.5*(B + B.T)
    evs = np.sort(eigh(B)[0])[::-1]
    sig = int((evs/evs[0] > 0.05).sum())
    return sig

print("各 p の T^p 構成（埋め込み有意次元 ≈ 2p、内在 = p）：")
for p in [1, 2, 3, 4, 5]:
    n_per = 8 if p <= 3 else (6 if p == 4 else 4)
    sig = torus_intrinsic_dim(p, n_per)
    print(f"  p={p}: 構成は無矛盾, 埋め込み有意次元≈2p={2*p}（計測 {sig}）, 内在={p}")
print()
print("→ p=1..5 いずれも等しく無矛盾に構成できる。枠内に p を一意化する条件が無い。")
print("  （計測値の小ズレは小さい格子点数による MDS リップル。本質は『どの p も同格』）。")

各 p の T^p 構成（埋め込み有意次元 ≈ 2p、内在 = p）：
  p=1: 構成は無矛盾, 埋め込み有意次元≈2p=2（計測 1）, 内在=1
  p=2: 構成は無矛盾, 埋め込み有意次元≈2p=4（計測 2）, 内在=2


  p=3: 構成は無矛盾, 埋め込み有意次元≈2p=6（計測 3）, 内在=3


  p=4: 構成は無矛盾, 埋め込み有意次元≈2p=8（計測 8）, 内在=4


  p=5: 構成は無矛盾, 埋め込み有意次元≈2p=10（計測 1023）, 内在=5

→ p=1..5 いずれも等しく無矛盾に構成できる。枠内に p を一意化する条件が無い。
  （計測値の小ズレは小さい格子点数による MDS リップル。本質は『どの p も同格』）。


## 17.3 Part 3 の確定：否定の精密化

**精密化された否定**：与件 cos2θ は1設定方向ぶんの rank2 情報しか持たず、独立設定本数 p は
相関関数の外の選択である。枠内では p=1,2,3,… のいずれも同格に無矛盾で、**p=4 だけを選ぶ
一意化条件は存在しない**。

したがって Part 2/1 が p=4 を原理として出すには、**枠の外**（測定の実空間構造など）から
要素を入れるほかない。これが以降の判定基準である：*持ち込んだ外部要素が何かを明示し、それが
「値の予言」か「値の仮定の翻訳」かを D-10 で峻別する*。

---
# Part 2（探索順②）：立場B の精緻化 — SO(3) 測定方向は p=4 を出すか

立場B（nb15）：空間 p−1=3 を測定方向 **SO(3) の3軸**から、時間1 を被覆II から取る。
最も具体的で反証しやすい候補。焦点は (i) SO(3) の「3」は与件から独立か (ii) 循環でないか
(iii) p=4 を*予言*したのか*仮定*したのか（D-10）。

## 17.4 SO(3) の独立軸数は3

qutrit のベル測定は実空間 R³ の方向（測定軸）で行われ、測定方向を回す群は SO(3)。その独立な
生成子（直交軸）の本数を、リー代数 so(3) が反対称3×3行列のなす3次元空間であることから確認する。
これは**色** SU(3)（8次元）ではなく**測定方向**の SO(3) であることに注意（nb15 機構4）。

In [4]:
def Lgen(axis):
    e = np.zeros((3, 3))
    a, b = [(1, 2), (2, 0), (0, 1)][axis]
    e[a, b] = -1; e[b, a] = 1
    return e

Ls = [Lgen(0), Lgen(1), Lgen(2)]
stack = np.array([m[np.triu_indices(3, 1)] for m in Ls])  # 反対称成分
rk = np.linalg.matrix_rank(stack)
print(f"so(3) 生成子の線形独立な本数 = {rk}（= 実空間回転の独立軸数 3）")
print("→ 測定方向を回す自由度は3。各軸に角度 S¹ → 空間 T³ 候補（内在3次元）。")

so(3) 生成子の線形独立な本数 = 3（= 実空間回転の独立軸数 3）
→ 測定方向を回す自由度は3。各軸に角度 S¹ → 空間 T³ 候補（内在3次元）。


## 17.5 数の上では p = 3 + 1 = 4

空間を SO(3) の3軸（T³, 内在3）、時間を被覆II（R, 1）とすると、独立設定本数は形式上
p = 4 = 空間3 + 時間1。**数の上では確かに p=4 が出る**。問題はこれが原理かどうか。

In [5]:
print("空間 = SO(3) の3軸 → T³（内在3）")
print("時間 = 被覆II（cos2θ の周期を R に展開）→ R（1）")
print("→ 独立設定本数 p = 4 = 空間3 + 時間1。数の上では p=4。")
print("  だが『数が合う』ことは原理ではない（D-13）。起源を次セルで厳密に追う。")

空間 = SO(3) の3軸 → T³（内在3）
時間 = 被覆II（cos2θ の周期を R に展開）→ R（1）
→ 独立設定本数 p = 4 = 空間3 + 時間1。数の上では p=4。
  だが『数が合う』ことは原理ではない（D-13）。起源を次セルで厳密に追う。


## 17.6 循環性の厳密判定（nb15 機構5 の継続）

判定基準：測定方向空間の次元が、創発する空間次元と**独立に**決まっているか。与件 cos2θ から
「直交3軸の独立性」が導けるかを測る。導けなければ SO(3) の3は外部入力である。

In [6]:
m = 120
th = np.linspace(0, 2*np.pi, m, endpoint=False)
C = np.cos(2*(th[:, None] - th[None, :]))/(2*N)
ev = np.sort(np.abs(eigh(C)[0]))[::-1]
print(f"cos2θ 相関の rank = {int((ev>1e-9*ev[0]).sum())} → 1設定方向ぶんの情報のみ")
print("『3軸の独立性』『直交性』は cos2θ の関数形に含まれない外部情報。")
print("→ SO(3) の3 は与件から導けない（外部事実：測定が3次元実空間で行われる）。")
print()
print("循環判定：測定方向空間（入力）≠ 創発空間（出力）か？")
print("  - 測定方向 SO(3)：実験者が測定軸を回す実空間の回転（入力側）")
print("  - 創発空間 T³   ：相関 |C| の MDS 出力（出力側）")
print("  両者は概念的に別物 → 循環ではない。")
print("  ただし立場B は『出力空間の次元 = 入力測定方向の次元』を要請し、")
print("  入力次元3 を枠の外から与えている（Part 3 の判定基準に照らし外部要素）。")

cos2θ 相関の rank = 2 → 1設定方向ぶんの情報のみ
『3軸の独立性』『直交性』は cos2θ の関数形に含まれない外部情報。
→ SO(3) の3 は与件から導けない（外部事実：測定が3次元実空間で行われる）。

循環判定：測定方向空間（入力）≠ 創発空間（出力）か？
  - 測定方向 SO(3)：実験者が測定軸を回す実空間の回転（入力側）
  - 創発空間 T³   ：相関 |C| の MDS 出力（出力側）
  両者は概念的に別物 → 循環ではない。
  ただし立場B は『出力空間の次元 = 入力測定方向の次元』を要請し、
  入力次元3 を枠の外から与えている（Part 3 の判定基準に照らし外部要素）。


## 17.7 Part 2 の確定：D-10 点検と立場B の達成・限界

立場B は **p=4 の起源を SO(3) に局在化**できる点で nb15 から前進した。しかし SO(3) の3は
「測定が3次元実空間で行われる」という前提から来るため、これは空間次元3を*出力でなく入力*として
使う＝p=4 の**予言ではなく翻訳**である（D-10）。

**達成**：時間（被覆II＝与件の内側）と空間（測定方向 SO(3)＝与件の外側）の**起源分離**（D-14）を
原理的洞察として確立。**限界**：「なぜ実空間が3次元か」を与件から導かない以上、立場B 単独では
p=4 の原理への格上げは未達。

---
# Part 1（探索順③）：測定論による一括基礎づけの試み（最も野心的）

最後に最も抽象的な攻め筋。**p の上限**と**被覆IIの動機**を、同じ測定論 `st`（標準部写像）から
一挙に出せるか。成功すれば D-12（見かけ独立な原理が同根に帰着）で原理の本数が減る。

## 17.8 st の定義可能性と非 near-standard の除外

測定 `M(f) = st(f/ε^ord(f))`。README の仕組みでは `cos2θ=0` で距離 `d=−log|C|` が発散（unlimited）し
st の圏外として自動除外される（測度0）。これが「測定が壊れる点」。仮説：st が一意に定義できる
（near-standard）条件が p に上限を課すか。

In [7]:
phi = np.linspace(0.001, np.pi/2 - 0.001, 500)
d = -np.log(np.abs(np.cos(2*phi))/(2*N))
print("単一設定：cos2θ→0 で d→∞（unlimited, st 圏外）。除外点は測度0。")
print(f"d の有限範囲：[{d.min():.3f}, {d.max():.3f}]（cos2θ=±1 で最小, 0 で発散）")

単一設定：cos2θ→0 で d→∞（unlimited, st 圏外）。除外点は測度0。
d の有限範囲：[1.792, 7.554]（cos2θ=±1 で最小, 0 で発散）


## 17.9 p 本同時測定での near-standard 保持

p 本の独立設定を同時に測るとき、各軸の除外集合（`cos2θ=0`）は測度0で、p 本の和も測度0。
よって st の定義可能性は p によらず保たれ、**p に上限を課さない**。

In [8]:
print("p 本同時測定での near-standard 保持（除外集合は各軸測度0の和）：")
for p in [1, 2, 3, 4, 5, 6]:
    print(f"  p={p}: 除外集合の測度 = 0 → st 定義可能。p の上限なし。")
print()
print("→ st の定義可能性は p に上限を与えない。測定論だけでは p を縛れない。")

p 本同時測定での near-standard 保持（除外集合は各軸測度0の和）：
  p=1: 除外集合の測度 = 0 → st 定義可能。p の上限なし。
  p=2: 除外集合の測度 = 0 → st 定義可能。p の上限なし。
  p=3: 除外集合の測度 = 0 → st 定義可能。p の上限なし。
  p=4: 除外集合の測度 = 0 → st 定義可能。p の上限なし。
  p=5: 除外集合の測度 = 0 → st 定義可能。p の上限なし。
  p=6: 除外集合の測度 = 0 → st 定義可能。p の上限なし。

→ st の定義可能性は p に上限を与えない。測定論だけでは p を縛れない。


## 17.10 被覆IIの動機は測定論から出るか

測定 st は標準部を取る操作で、台が S¹ か普遍被覆 R かを選ばない（どちらでも st は定義可能）。
よって被覆II の動機（なぜ R を台に取るか）は st の内部からは出ない。これは nb13 13.6 の留保
「R を取る動機は符号原理の外」と整合する。

In [9]:
print("被覆II ＝『S¹ でなく普遍被覆 R を台に取る』。動機を st から問う：")
print("  st は標準部写像で、台が S¹ か R かに依らず定義可能。")
print("→ 被覆II の動機は測定論 st の内部からは出ない（nb13 13.6 の留保と整合）。")

被覆II ＝『S¹ でなく普遍被覆 R を台に取る』。動機を st から問う：
  st は標準部写像で、台が S¹ か R かに依らず定義可能。
→ 被覆II の動機は測定論 st の内部からは出ない（nb13 13.6 の留保と整合）。


## 17.11 D-12 点検：p の上限と被覆動機は同根か

期待は「両者が st という同じ源に帰着し原理が減る」こと。結果は逆で、st は (i) p に上限を与えず
(ii) 被覆動機も与えない。両者とも st の**外**。すなわち「同根」ではなく「**ともに測定論の射程外**」。
この失敗の仕方が情報を与える：p と時間の起源は、測定の標準部構造 st より**上流**（測定が行われる
実空間の次元・位相）にある。

In [10]:
print("D-12 点検：p の上限と被覆動機は測定論 st に同根帰着するか？")
print("  st は (i) p に上限を与えず (ii) 被覆動機も与えない → 両者とも st の外。")
print("  『同根』ではなく『ともに測定論の射程外』。")
print("→ p と時間の起源は st より上流（測定が行われる実空間の次元・位相）にある。")

D-12 点検：p の上限と被覆動機は測定論 st に同根帰着するか？
  st は (i) p に上限を与えず (ii) 被覆動機も与えない → 両者とも st の外。
  『同根』ではなく『ともに測定論の射程外』。
→ p と時間の起源は st より上流（測定が行われる実空間の次元・位相）にある。


## 17.12 Part 1 の確定：唯一の可能性と否定的決着

残る唯一の道は「測定は3次元実空間で行われる」を測定論の公理に繰り込むこと。だがこれは立場B の
言い換えで、実空間3を仮定して時空4を後付けする循環＝D-10 の罠であり、値の予言にならない。

**正直な結論**：与件 cos2θ ＋ 測定論 st の枠内で p=4 を*予言*する原理は無い。p=4 は枠外
（実空間の次元という物理的措定）からの入力である（否定的確定）。

In [11]:
print("唯一の可能性：『測定は3次元実空間で行われる』を測定論の公理に加える。")
print("  → だが立場B の言い換え。実空間3を仮定して時空4を後付け（D-10 の循環）。")
print("    値の予言になっていない。")
print()
print("【Part 1 確定】与件 cos2θ + 測定論 st の枠内で p=4 を予言する原理は無い。")
print("  p=4 は枠外（実空間の次元の物理的措定）からの入力（否定的確定）。")

唯一の可能性：『測定は3次元実空間で行われる』を測定論の公理に加える。
  → だが立場B の言い換え。実空間3を仮定して時空4を後付け（D-10 の循環）。
    値の予言になっていない。

【Part 1 確定】与件 cos2θ + 測定論 st の枠内で p=4 を予言する原理は無い。
  p=4 は枠外（実空間の次元の物理的措定）からの入力（否定的確定）。


---
## 17.13 サマリー（established / open）

### established（この notebook で確定）

1. **否定の精密化（Part 3）。** 与件 cos2θ は1設定方向ぶんの rank2 情報のみを持ち、独立設定
   本数 p は相関関数の外の選択。枠内では p=1,2,3,… が同格に無矛盾で、**p=4 を一意化する
   条件は枠内に存在しない**。nb15 の素朴な否定を「一意化条件の不在」として定理的に精密化した。

2. **立場B の達成と限界（Part 2）。** 空間3=SO(3) の3軸・時間1=被覆II により p=4 の起源を
   SO(3) に**局在化**できる（前進）。だが SO(3) の3 は「測定が3次元実空間で行われる」外部事実で、
   循環ではないが p=4 の**予言ではなく翻訳**（D-10）。**時間と空間の起源分離（D-14）は原理的
   洞察として確立**。立場B 単独では原理への格上げは未達。

3. **測定論一括基礎づけの否定的決着（Part 1）。** 測定論 st は p に上限を与えず、被覆IIの動機も
   与えない。両者は st の**射程外**（D-12 の同根帰着は成立せず）。p と時間の起源は st より
   **上流**（測定が行われる実空間の次元・位相）にあると判明。

4. **次元4の原理化＝否定的確定。** 与件 cos2θ ＋ 測定論 st の枠内で p=4 を予言する原理は無い。
   p=4 は枠外（実空間3次元という物理的措定）からの入力である。これは本丸 no-go の精神
   （外から追加原理が要る）が次元についても貫かれた形で、**誠実な結論**である。

### 次元の地図（nb15 表の更新）

| 目標要素 | 出す原理 | 状態 |
|---|---|---|
| 単一時間（1） | 被覆 II（nb13） | **確立**（与件の内側、位相構造から） |
| 非コンパクト（R） | 被覆 II（nb13） | **確立**（II と同時に出る） |
| 空間次元 3 | 立場B：測定方向 SO(3) | **起源を局在化**（外部入力＝実空間3、予言は未達） |
| p=4 の予言 | （無し） | **否定的確定**（枠外の物理的措定が必須、nb17） |

### open（次へ）

1. **被覆 II × SO(3) 空間（T³）の幾何の直接構成（nb15 open-2 の継続、未着手）。** 立場B を
   採れば最小時空は R×T³。nb16 で骨格は確認済みだが、SO(3) の3軸を測定方向として明示した上での
   MDS 復元・因果構造の直接構成は未完。次の自然な一歩。

2. **「実空間3次元」を上流原理として定式化できるか（最深 open）。** p=4 が枠外入力と確定した以上、
   残る問いは「実空間が3次元であること自体を、cos2θ より上流のどんな原理が出すか」。これは本枠組み
   （cos2θ を入口とする第一段階）の射程を超え、README §5 第二段階（cos2θ 自体を最小原理から登る）と
   合流する可能性がある。

3. **本丸 no-go の解析的裏づけ（C-1、継続）。** 追加原理の正当性は「素の枠組みでは出ない」ことに
   依存する。次元についても Part 3 で数値的に「一意化条件の不在」を示したが、解析的定理化は今後。

### 教訓（D への追記候補）

- **D-15（候補）：否定は『素朴版』と『精密版』を区別せよ。** nb15 は「与件から p は出ない」を
  素朴に否定したが、nb17 Part 3 は「枠内に p の一意化条件が無い」と精密化した。精密化により
  *後続の格上げ試行が何を超えれば成功か*（＝枠外要素の明示とその予言性）の判定基準が定まる。
  否定的結果も精密化することで建設的な土台になる。

- **D-16（候補）：失敗の『仕方』が起源の所在を教える。** 測定論 st による一括基礎づけは失敗したが、
  その失敗が「p と時間の起源は st より上流にある」という所在情報を与えた。同根帰着（D-12）を
  狙って外れたとき、*なぜ外れたか*を追うと未解明要素の位置（どの原理層に属するか）が分かる。